# Build `questions_no_hint.csv`

Notebook per trasformare `data/questions.csv` in un CSV senza opzioni multiple-choice e con risposta testuale corretta.


## 0) Setup percorsi e import


In [1]:
from pathlib import Path
import re

import pandas as pd

ROOT = Path("/Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite")
INPUT_CSV = ROOT / "data" / "questions.csv"
OUTPUT_CSV = ROOT / "data" / "questions_no_hint.csv"

print("Input:", INPUT_CSV)
print("Output:", OUTPUT_CSV)
print("Input exists:", INPUT_CSV.exists())


Input: /Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/data/questions.csv
Output: /Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/data/questions_no_hint.csv
Input exists: True


## 1) Load CSV originale


In [2]:
df = pd.read_csv(INPUT_CSV, encoding="utf-8")
print("Shape originale:", df.shape)
df.head(3)


Shape originale: (163, 5)


,#,Domanda,Livello,Risposta corretta,Riferimento legge per la risposta
0,1,Quali sono gli organi dell'azienda Usl?\nA) Il...,L1,D,"Legge regionale 25 gennaio 2000, n. 5 - Art 12\n"
1,2,Come si attua la programmazione sanitaria regi...,L2,E,"Legge regionale 25 gennaio 2000, n. 5 - Art. 2"
2,3,Cosa accade in caso di vacanza dell'ufficio o ...,L3,D,"Legge regionale 25 gennaio 2000, n. 5 - Art.16"


## 2) Regex helpers


In [3]:
OPTION_RE = re.compile(r"(?m)^\s*([A-F])\)\s+(.*)$") # Opzioni etichettate da A) a F), con testo dopo la parentesi. Multilinea per catturare tutte le opzioni.
EXPECTED_LABELS = ["A", "B", "C", "D", "E", "F"]

def _safe_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value)

def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def extract_stem_and_options(domanda: str) -> tuple[str, dict[str, str]]:
    domanda = _safe_text(domanda).strip("\n")
    matches = list(OPTION_RE.finditer(domanda))
    if not matches:
        raise ValueError("No options found in `Domanda` field")

    stem = normalize_whitespace(domanda[: matches[0].start()])
    options = {m.group(1).strip(): normalize_whitespace(m.group(2)) for m in matches}

    labels = list(options.keys())
    if labels != EXPECTED_LABELS:
        raise ValueError(f"Unexpected option labels/order: {labels!r}")

    return stem, options

def normalize_references(reference_text: str) -> str:
    parts = [
        normalize_whitespace(line)
        for line in _safe_text(reference_text).splitlines()
        if line and line.strip()
    ]
    return " | ".join(parts)

def map_answer(row: pd.Series) -> dict[str, str]:
    qid = normalize_whitespace(_safe_text(row.get("#", ""))) or "<missing>"

    stem, options = extract_stem_and_options(_safe_text(row.get("Domanda", "")))

    answer_label = normalize_whitespace(_safe_text(row.get("Risposta corretta", ""))).upper()
    if answer_label not in options:
        raise ValueError(f"Q{qid}: answer label {answer_label!r} not found in options")

    return {
        "Domanda": stem,
        "Livello": normalize_whitespace(_safe_text(row.get("Livello", ""))),
        "Risposta corretta": options[answer_label],
        "Riferimento legge per la risposta": normalize_references(row.get("Riferimento legge per la risposta", "")),
    }


## 3) Transform


In [4]:
# Escludo righe vuote finali e qualunque riga senza risposta corretta.
mask_valid = df["Risposta corretta"].notna() & (df["Risposta corretta"].astype(str).str.strip() != "")
df_valid = df.loc[mask_valid].copy()

print("Righe valide:", len(df_valid))

records = [map_answer(row) for _, row in df_valid.iterrows()]
df_out = pd.DataFrame(
    records,
    columns=["Domanda", "Livello", "Risposta corretta", "Riferimento legge per la risposta"],
)

print("Shape output:", df_out.shape)
df_out.head(3)


Righe valide: 100
Shape output: (100, 4)


,Domanda,Livello,Risposta corretta,Riferimento legge per la risposta
0,Quali sono gli organi dell'azienda Usl?,L1,Il direttore generale il collegio sindacale,"Legge regionale 25 gennaio 2000, n. 5 - Art 12"
1,Come si attua la programmazione sanitaria regi...,L2,Attraverso il Piano socio-sanitario regionale,"Legge regionale 25 gennaio 2000, n. 5 - Art. 2"
2,Cosa accade in caso di vacanza dell'ufficio o ...,L3,Le relative funzioni sono svolte dal direttore...,"Legge regionale 25 gennaio 2000, n. 5 - Art.16"


## 4) Validation


In [5]:
assert list(df_out.columns) == [
    "Domanda",
    "Livello",
    "Risposta corretta",
    "Riferimento legge per la risposta",
]

assert not df_out.isna().any().any(), "Sono presenti NaN nell'output"
assert len(df_out) == 100, f"Numero righe inatteso: {len(df_out)}"

expected_levels = {"L1": 25, "L2": 25, "L3": 25, "L4": 25}
actual_levels = df_out["Livello"].value_counts().sort_index().to_dict()
assert actual_levels == expected_levels, f"Distribuzione livelli inattesa: {actual_levels}"

print("Distribuzione livelli:", actual_levels)

sample_indices = [0, len(df_out) // 2, len(df_out) - 1]
df_out.iloc[sample_indices]


Distribuzione livelli: {'L1': 25, 'L2': 25, 'L3': 25, 'L4': 25}


,Domanda,Livello,Risposta corretta,Riferimento legge per la risposta
0,Quali sono gli organi dell'azienda Usl?,L1,Il direttore generale il collegio sindacale,"Legge regionale 25 gennaio 2000, n. 5 - Art 12"
50,Com’è disciplinato l’uso di rilevatori di meta...,L3,L’uso di rilevatori di metalli è vietato salva...,"Legge regionale 10 giugno 1983, n. 56 - Art. 7"
99,Con quali provvedimenti la Regione viene in ai...,L4,Attraverso integrazioni alle sovvenzioni previ...,"Legge regionale 21 aprile 2020, n. 5 - Art. 5"


## 5) Write output CSV


In [6]:
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print("File scritto:", OUTPUT_CSV)
print("Esiste:", OUTPUT_CSV.exists())


File scritto: /Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/data/questions_no_hint.csv
Esiste: True


## 6) Preview output


In [7]:
df_preview = pd.read_csv(OUTPUT_CSV, encoding="utf-8")
print("Shape riletta:", df_preview.shape)
df_preview.head(10)


Shape riletta: (100, 4)


,Domanda,Livello,Risposta corretta,Riferimento legge per la risposta
0,Quali sono gli organi dell'azienda Usl?,L1,Il direttore generale il collegio sindacale,"Legge regionale 25 gennaio 2000, n. 5 - Art 12"
1,Come si attua la programmazione sanitaria regi...,L2,Attraverso il Piano socio-sanitario regionale,"Legge regionale 25 gennaio 2000, n. 5 - Art. 2"
2,Cosa accade in caso di vacanza dell'ufficio o ...,L3,Le relative funzioni sono svolte dal direttore...,"Legge regionale 25 gennaio 2000, n. 5 - Art.16"
3,Quali istituzioni partecipano alla politica so...,L4,Gli enti locali,"Legge regionale 25 gennaio 2000, n. 5 - Art. 3"
4,Da quanti membri è composta la Consulta region...,L1,Da 7 membri,"Legge regionale 8 ottobre 2019, n. 16 - Art. 4..."
5,Con quali dei seguenti strumenti la Regione in...,L2,La Giunta regionale s'impegna al completamento...,"Legge regionale 8 ottobre 2019, n. 16 - Art. 4"
6,"Io ho 32 anni, sono un soggetto privato che no...",L3,14.000 euro,"Legge regionale 8 ottobre 2019, n. 16 - Art. 8"
7,Io ho comprato un'automobile nel 2025 benefici...,L4,Revoca del contributo e obbligo di restituire ...,"Legge regionale 8 ottobre 2019, n. 16 - Art. 13"
8,Qual è lo strumento per la pianificazione urba...,L1,Il Piano regolatore generale comunale urbanist...,"Legge regionale 6 aprile 1998, n. 11 - Art. 11..."
9,Per effettuare una trasformazione urbanistica ...,L2,Di un permesso di costruire o altro titolo abi...,"Legge regionale 6 aprile 1998, n. 11 - Art. 59"
